# ALMA Archive Connection

This notebook explores programmatic access to the ALMA Science Archive.

## Initial objectives

1. Verify the Python and Jupyter environment.
2. Connect to the ALMA Science Archive.
3. Run a minimal archive query.
4. Inspect the returned columns and units.
5. Identify fields required for duplication checking.

In [1]:
import sys
from pathlib import Path

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nCurrent working directory:")
print(Path.cwd())

Python executable:
/Users/nana/opt/anaconda3/envs/alma-duplication/bin/python

Python version:
3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 08:06:15) [Clang 14.0.6 ]

Current working directory:
/Users/nana/Documents/alma-duplication-tool/notebooks


In [2]:
import astropy
import astroquery
import pyvo
import pandas as pd
import matplotlib

print("Astropy:", astropy.__version__)
print("Astroquery:", astroquery.__version__)
print("PyVO:", pyvo.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)

print("\nNotebook environment is ready.")

Astropy: 8.0.1
Astroquery: 0.4.11
PyVO: 1.9.1
Pandas: 3.0.5
Matplotlib: 3.11.1

Notebook environment is ready.


In [3]:
from astropy.coordinates import SkyCoord
import astropy.units as u

test_coordinate = SkyCoord(
    ra="12h34m56.7s",
    dec="-12d34m56s",
    frame="icrs",
)

print("RA in degrees:", test_coordinate.ra.deg)
print("Dec in degrees:", test_coordinate.dec.deg)
print("Coordinate:", test_coordinate)

RA in degrees: 188.73624999999998
Dec in degrees: -12.582222222222223
Coordinate: <SkyCoord (ICRS): (ra, dec) in deg
    (188.73625, -12.58222222)>


## Environment check

- Python environment: `alma-duplication`
- Python version: 3.12.11
- Archive access package: PyVO
- Coordinate handling package: Astropy
- Table-processing package: Pandas
- Status: Development environment successfully configured

## Next task

Connect to the ALMA Science Archive TAP service and run a minimal query.

In [1]:
import alma_duplicate

print("Package location:")
print(alma_duplicate.__file__)

print("\nLocal project package imported successfully.")

Package location:
/Users/nana/Documents/alma-duplication-tool/src/alma_duplicate/__init__.py

Local project package imported successfully.


## ALMA TAP service connection

The European ALMA Science Archive provides a Table Access Protocol
(TAP) service. Queries are written in the Astronomical Data Query
Language (ADQL) and executed using PyVO.

This section:

1. Creates a connection to the European ALMA TAP service.
2. Executes a small query with a maximum of five rows.
3. Inspects the returned columns and data types.

In [2]:
import pandas as pd
import pyvo

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

TAP_URL = "https://almascience.eso.org/tap"

print("PyVO version:", pyvo.__version__)
print("TAP endpoint:", TAP_URL)

PyVO version: 1.9.1
TAP endpoint: https://almascience.eso.org/tap


In [3]:
service = pyvo.dal.TAPService(TAP_URL)

print("TAP service object created successfully.")
print(service)

TAP service object created successfully.
TAPService(baseurl : 'https://almascience.eso.org/tap', description : 'None')


### Minimal archive query

The first query requests only five rows and a limited number of columns.
This verifies the connection without downloading a large result table.

In [4]:
minimal_query = """
SELECT TOP 5
    proposal_id,
    target_name,
    s_ra,
    s_dec,
    frequency,
    bandwidth,
    spatial_resolution,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    member_ous_uid,
    is_mosaic
FROM ivoa.obscore
WHERE science_observation = 'T'
"""

print(minimal_query)


SELECT TOP 5
    proposal_id,
    target_name,
    s_ra,
    s_dec,
    frequency,
    bandwidth,
    spatial_resolution,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    member_ous_uid,
    is_mosaic
FROM ivoa.obscore
WHERE science_observation = 'T'



In [5]:
response = service.search(minimal_query)

print("Query completed successfully.")
print("Response type:", type(response))

Query completed successfully.
Response type: <class 'pyvo.dal.tap.TAPResults'>


In [6]:
archive_table = response.to_table()

print("Astropy table created.")
print("Number of rows:", len(archive_table))
print("Number of columns:", len(archive_table.colnames))
print("Columns:")
print(archive_table.colnames)

Astropy table created.
Number of rows: 5
Number of columns: 11
Columns:
['proposal_id', 'target_name', 's_ra', 's_dec', 'frequency', 'bandwidth', 'spatial_resolution', 'sensitivity_10kms', 'cont_sensitivity_bandwidth', 'member_ous_uid', 'is_mosaic']


In [7]:
archive_df = archive_table.to_pandas()

print("DataFrame shape:", archive_df.shape)
display(archive_df)

DataFrame shape: (5, 11)


,proposal_id,target_name,s_ra,s_dec,frequency,bandwidth,spatial_resolution,sensitivity_10kms,cont_sensitivity_bandwidth,member_ous_uid,is_mosaic
0,2021.1.00869.L,ad3a-22078,240.052125,-54.425472,87.492595,9.375000e+08,1.159672,3.811278,0.105020,uid://A001/X1590/X18e0,F
1,2021.1.00869.L,ad3a-22078,240.052125,-54.425472,84.566616,9.375000e+08,1.159672,3.783308,0.105020,uid://A001/X1590/X18e0,F
2,2021.1.00869.L,ad3a-22078,240.052125,-54.425472,85.398372,9.375000e+08,1.159672,3.765423,0.105020,uid://A001/X1590/X18e0,F
3,2021.1.00869.L,ad3a-22078,240.052125,-54.425472,86.545116,9.375000e+08,1.159672,3.831390,0.105020,uid://A001/X1590/X18e0,F
4,2024.1.01553.S,3C078,47.109262,4.110915,225.732519,2.000000e+09,0.509148,0.786219,0.026225,uid://A001/X3788/Xb661,F


In [8]:
print("Column data types:")
display(archive_df.dtypes.to_frame(name="dtype"))

print("\nMissing values:")
display(archive_df.isna().sum().to_frame(name="missing_count"))

Column data types:


,dtype
proposal_id,str
target_name,str
s_ra,float64
s_dec,float64
frequency,float64
bandwidth,float64
spatial_resolution,float64
sensitivity_10kms,float64
cont_sensitivity_bandwidth,float64
member_ous_uid,str



Missing values:


,missing_count
proposal_id,0
target_name,0
s_ra,0
s_dec,0
frequency,0
bandwidth,0
spatial_resolution,0
sensitivity_10kms,0
cont_sensitivity_bandwidth,0
member_ous_uid,0


In [9]:
column_metadata = []

for column_name in archive_table.colnames:
    column = archive_table[column_name]

    column_metadata.append(
        {
            "column": column_name,
            "unit": str(column.unit) if column.unit is not None else None,
            "dtype": str(column.dtype),
        }
    )

metadata_df = pd.DataFrame(column_metadata)

display(metadata_df)

,column,unit,dtype
0,proposal_id,NaN,<U64
1,target_name,NaN,<U256
2,s_ra,deg,float64
3,s_dec,deg,float64
4,frequency,GHz,float64
5,bandwidth,Hz,float64
6,spatial_resolution,arcsec,float64
7,sensitivity_10kms,mJy / beam,float64
8,cont_sensitivity_bandwidth,mJy / beam,float64
9,member_ous_uid,NaN,<U64


## Coordinate-based Archive query

This section queries Archive records whose spatial footprints intersect
a circular search region.

The initial example follows the official ALMA single-source query notebook
and searches around the position of Centaurus A.

The query includes additional identifiers to investigate how Archive rows,
spectral windows, datasets, and Member OUS records are related.

In [10]:
from astropy.coordinates import SkyCoord
import astropy.units as u

search_coordinate = SkyCoord(
    ra=201.365 * u.deg,
    dec=-43.019 * u.deg,
    frame="icrs",
)

search_radius = 21.6 * u.arcsec

ra_deg = search_coordinate.ra.deg
dec_deg = search_coordinate.dec.deg
radius_deg = search_radius.to(u.deg).value

print("Search RA:", ra_deg, "deg")
print("Search Dec:", dec_deg, "deg")
print("Search radius:", search_radius)
print("Search radius:", radius_deg, "deg")

Search RA: 201.365 deg
Search Dec: -43.019 deg
Search radius: 21.6 arcsec
Search radius: 0.006 deg


In [11]:
coordinate_query = f"""
SELECT TOP 100
    proposal_id,
    group_ous_uid,
    member_ous_uid,
    obs_id,
    asdm_uid,
    target_name,
    s_ra,
    s_dec,
    s_region,
    frequency,
    bandwidth,
    frequency_support,
    spatial_resolution,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    antenna_arrays,
    is_mosaic
FROM ivoa.obscore
WHERE science_observation = 'T'
AND INTERSECTS(
    CIRCLE('ICRS', {ra_deg}, {dec_deg}, {radius_deg}),
    s_region
) = 1
"""

print(coordinate_query)


SELECT TOP 100
    proposal_id,
    group_ous_uid,
    member_ous_uid,
    obs_id,
    asdm_uid,
    target_name,
    s_ra,
    s_dec,
    s_region,
    frequency,
    bandwidth,
    frequency_support,
    spatial_resolution,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    antenna_arrays,
    is_mosaic
FROM ivoa.obscore
WHERE science_observation = 'T'
AND INTERSECTS(
    CIRCLE('ICRS', 201.365, -43.019, 0.006),
    s_region
) = 1



In [12]:
coordinate_response = service.search(coordinate_query)
coordinate_table = coordinate_response.to_table()
coordinate_df = coordinate_table.to_pandas()

print("Coordinate query completed successfully.")
print("Number of rows:", len(coordinate_df))
print("Number of columns:", len(coordinate_df.columns))

Coordinate query completed successfully.
Number of rows: 100
Number of columns: 17


In [13]:
display_columns = [
    "proposal_id",
    "member_ous_uid",
    "obs_id",
    "target_name",
    "s_ra",
    "s_dec",
    "frequency",
    "bandwidth",
    "spatial_resolution",
    "is_mosaic",
]

display(coordinate_df[display_columns].head(20))

,proposal_id,member_ous_uid,obs_id,target_name,s_ra,s_dec,frequency,bandwidth,spatial_resolution,is_mosaic
0,2012.1.00225.S,uid://A002/X7d1738/Xf8,uid://A002/X7d1738/Xf8.source.Centaurus_a.spw.19,Centaurus_a,201.365069,-43.018989,707.588542,1.875000e+09,0.174587,T
1,2012.1.00225.S,uid://A002/X7d1738/Xf8,uid://A002/X7d1738/Xf8.source.Centaurus_a.spw.17,Centaurus_a,201.365069,-43.018989,690.215941,1.875000e+09,0.174587,T
2,2012.1.00225.S,uid://A002/X7d1738/Xf8,uid://A002/X7d1738/Xf8.source.Centaurus_a.spw.23,Centaurus_a,201.365069,-43.018989,703.921586,1.875000e+09,0.174587,T
3,2012.1.00225.S,uid://A002/X7d1738/Xf8,uid://A002/X7d1738/Xf8.source.Centaurus_a.spw.21,Centaurus_a,201.365069,-43.018989,692.039544,1.875000e+09,0.174587,T
4,2011.0.00010.S,uid://A002/X327408/X213,uid://A002/X327408/X213.source.CenA.spw.23,CenA,201.365063,-43.019112,87.767100,2.343750e+08,1.495708,F
5,2011.0.00010.S,uid://A002/X327408/X213,uid://A002/X327408/X213.source.CenA.spw.17,CenA,201.365063,-43.019112,89.028119,2.343750e+08,1.495708,F
6,2011.0.00010.S,uid://A002/X327408/X213,uid://A002/X327408/X213.source.CenA.spw.19,CenA,201.365063,-43.019112,88.472196,2.343750e+08,1.495708,F
7,2011.0.00010.S,uid://A002/X327408/X213,uid://A002/X327408/X213.source.CenA.spw.21,CenA,201.365063,-43.019112,87.171528,2.343750e+08,1.495708,F
8,2022.1.00506.S,uid://A001/X2d20/X26a4,uid://A001/X2d20/X26a4.source.Centaurus_A.spw.11,Centaurus_A,201.365017,-43.019304,151.988515,2.000000e+09,0.146130,F
9,2022.1.00506.S,uid://A001/X2d20/X26a4,uid://A001/X2d20/X26a4.source.Centaurus_A.spw.9,Centaurus_A,201.365017,-43.019304,150.030392,2.000000e+09,0.146130,F


In [14]:
identifier_summary = pd.DataFrame(
    {
        "field": [
            "proposal_id",
            "group_ous_uid",
            "member_ous_uid",
            "obs_id",
            "asdm_uid",
            "target_name",
        ],
        "unique_values": [
            coordinate_df["proposal_id"].nunique(dropna=True),
            coordinate_df["group_ous_uid"].nunique(dropna=True),
            coordinate_df["member_ous_uid"].nunique(dropna=True),
            coordinate_df["obs_id"].nunique(dropna=True),
            coordinate_df["asdm_uid"].nunique(dropna=True),
            coordinate_df["target_name"].nunique(dropna=True),
        ],
    }
)

display(identifier_summary)

,field,unique_values
0,proposal_id,11
1,group_ous_uid,17
2,member_ous_uid,25
3,obs_id,100
4,asdm_uid,25
5,target_name,5


In [15]:
member_ous_summary = (
    coordinate_df
    .groupby(
        ["proposal_id", "member_ous_uid", "target_name"],
        dropna=False,
    )
    .agg(
        row_count=("frequency", "size"),
        unique_frequencies=("frequency", "nunique"),
        min_frequency_ghz=("frequency", "min"),
        max_frequency_ghz=("frequency", "max"),
        unique_obs_ids=("obs_id", "nunique"),
        unique_asdm_uids=("asdm_uid", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["row_count", "unique_frequencies"],
        ascending=False,
    )
)

display(member_ous_summary.head(20))

,proposal_id,member_ous_uid,target_name,row_count,unique_frequencies,min_frequency_ghz,max_frequency_ghz,unique_obs_ids,unique_asdm_uids
0,2011.0.00008.SV,uid://A002/X259150/X157,Centaurus A,4,4,230.137724,246.385449,4,1
1,2011.0.00010.S,uid://A002/X327408/X20f,CenA,4,4,86.075216,99.236139,4,1
2,2011.0.00010.S,uid://A002/X327408/X211,CenA,4,4,109.592329,113.242400,4,1
3,2011.0.00010.S,uid://A002/X327408/X213,CenA,4,4,87.171528,89.028119,4,1
4,2011.0.00010.S,uid://A002/X327408/X215,CenA,4,4,90.500536,93.005842,4,1
5,2011.0.00010.S,uid://A002/X327408/X217,CenA,4,4,218.060550,219.990105,4,1
6,2012.1.00225.S,uid://A002/X7d1738/Xf8,Centaurus_a,4,4,690.215941,707.588542,4,1
7,2016.1.01198.V,uid://A001/X11a7/X14,Centaurus_A,4,4,213.096375,229.096103,4,1
8,2017.1.01162.S,uid://A001/X1284/X76,Centaurus_a,4,4,343.753036,356.085324,4,1
9,2017.1.01162.S,uid://A001/X12a3/Xb0,Centaurus_a,4,4,343.752909,356.085424,4,1


In [16]:
print("Mosaic values:")
display(
    coordinate_df["is_mosaic"]
    .value_counts(dropna=False)
    .to_frame(name="row_count")
)

print("\nMissing values:")
display(
    coordinate_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="missing_count")
)

Mosaic values:


,row_count
is_mosaic,
F,84
T,16



Missing values:


,missing_count
proposal_id,0
frequency,0
antenna_arrays,0
cont_sensitivity_bandwidth,0
sensitivity_10kms,0
spatial_resolution,0
frequency_support,0
bandwidth,0
s_region,0
group_ous_uid,0


In [17]:
coordinate_display_df = coordinate_df.copy()

coordinate_display_df["bandwidth_ghz"] = (
    coordinate_display_df["bandwidth"] / 1e9
)

display(
    coordinate_display_df[
        [
            "proposal_id",
            "member_ous_uid",
            "target_name",
            "frequency",
            "bandwidth",
            "bandwidth_ghz",
        ]
    ].head(20)
)

,proposal_id,member_ous_uid,target_name,frequency,bandwidth,bandwidth_ghz
0,2012.1.00225.S,uid://A002/X7d1738/Xf8,Centaurus_a,707.588542,1.875000e+09,1.875000
1,2012.1.00225.S,uid://A002/X7d1738/Xf8,Centaurus_a,690.215941,1.875000e+09,1.875000
2,2012.1.00225.S,uid://A002/X7d1738/Xf8,Centaurus_a,703.921586,1.875000e+09,1.875000
3,2012.1.00225.S,uid://A002/X7d1738/Xf8,Centaurus_a,692.039544,1.875000e+09,1.875000
4,2011.0.00010.S,uid://A002/X327408/X213,CenA,87.767100,2.343750e+08,0.234375
5,2011.0.00010.S,uid://A002/X327408/X213,CenA,89.028119,2.343750e+08,0.234375
6,2011.0.00010.S,uid://A002/X327408/X213,CenA,88.472196,2.343750e+08,0.234375
7,2011.0.00010.S,uid://A002/X327408/X213,CenA,87.171528,2.343750e+08,0.234375
8,2022.1.00506.S,uid://A001/X2d20/X26a4,Centaurus_A,151.988515,2.000000e+09,2.000000
9,2022.1.00506.S,uid://A001/X2d20/X26a4,Centaurus_A,150.030392,2.000000e+09,2.000000


### Count all matching Archive rows

The exploratory query returned exactly 100 rows and may therefore have
reached the `TOP 100` limit. Before downloading the complete result set,
a count query is used to determine the total number of matching Archive rows.

In [18]:
coordinate_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND INTERSECTS(
    CIRCLE('ICRS', {ra_deg}, {dec_deg}, {radius_deg}),
    s_region
) = 1
"""

print(coordinate_count_query)


SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND INTERSECTS(
    CIRCLE('ICRS', 201.365, -43.019, 0.006),
    s_region
) = 1



In [19]:
count_response = service.search(coordinate_count_query)
count_table = count_response.to_table()
count_df = count_table.to_pandas()

display(count_df)

,total_rows
0,434


In [20]:
total_matching_rows = int(count_df.iloc[0, 0])

print("Total matching Archive rows:", total_matching_rows)
print("Rows retrieved in exploratory query:", len(coordinate_df))
print(
    "Exploratory result was truncated:",
    total_matching_rows > len(coordinate_df),
)

Total matching Archive rows: 434
Rows retrieved in exploratory query: 100
Exploratory result was truncated: True


In [21]:
mosaic_summary = (
    coordinate_df
    .groupby("is_mosaic", dropna=False)
    .agg(
        archive_row_count=("obs_id", "size"),
        proposal_count=("proposal_id", "nunique"),
        member_ous_count=("member_ous_uid", "nunique"),
        asdm_count=("asdm_uid", "nunique"),
    )
    .reset_index()
)

display(mosaic_summary)

,is_mosaic,archive_row_count,proposal_count,member_ous_count,asdm_count
0,F,84,9,21,21
1,T,16,4,4,4


In [22]:
example_member_ous = coordinate_df["member_ous_uid"].iloc[0]

print("Example Member OUS:")
print(example_member_ous)

Example Member OUS:
uid://A002/X7d1738/Xf8


In [23]:
example_spectral_setup = (
    coordinate_df.loc[
        coordinate_df["member_ous_uid"] == example_member_ous,
        [
            "obs_id",
            "frequency",
            "bandwidth",
            "frequency_support",
            "sensitivity_10kms",
            "cont_sensitivity_bandwidth",
        ],
    ]
    .sort_values("frequency")
)

display(example_spectral_setup)

,obs_id,frequency,bandwidth,frequency_support,sensitivity_10kms,cont_sensitivity_bandwidth
1,uid://A002/X7d1738/Xf8.source.Centaurus_a.spw.17,690.215941,1.875000e+09,"[689.28..691.15GHz,976.56kHz,10.8mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [691.10..692.98GHz...",10.803344,0.593166
3,uid://A002/X7d1738/Xf8.source.Centaurus_a.spw.21,692.039544,1.875000e+09,"[689.28..691.15GHz,976.56kHz,10.8mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [691.10..692.98GHz...",10.819007,0.593166
2,uid://A002/X7d1738/Xf8.source.Centaurus_a.spw.23,703.921586,1.875000e+09,"[689.28..691.15GHz,976.56kHz,10.8mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [691.10..692.98GHz...",10.463668,0.593166
0,uid://A002/X7d1738/Xf8.source.Centaurus_a.spw.19,707.588542,1.875000e+09,"[689.28..691.15GHz,976.56kHz,10.8mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [691.10..692.98GHz...",10.496009,0.593166


## Retrieve the complete coordinate-query result

The exploratory query used `TOP 100` and returned only part of the result.
This section retrieves all matching Archive rows and verifies the result
against the previous `COUNT(*)` query.

In [24]:
full_coordinate_query = coordinate_query.replace(
    "SELECT TOP 100",
    "SELECT",
    1,
)

print(full_coordinate_query)


SELECT
    proposal_id,
    group_ous_uid,
    member_ous_uid,
    obs_id,
    asdm_uid,
    target_name,
    s_ra,
    s_dec,
    s_region,
    frequency,
    bandwidth,
    frequency_support,
    spatial_resolution,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    antenna_arrays,
    is_mosaic
FROM ivoa.obscore
WHERE science_observation = 'T'
AND INTERSECTS(
    CIRCLE('ICRS', 201.365, -43.019, 0.006),
    s_region
) = 1



In [25]:
full_response = service.search(
    full_coordinate_query,
    maxrec=total_matching_rows,
)

full_table = full_response.to_table()
full_coordinate_df = full_table.to_pandas()

print("Expected rows:", total_matching_rows)
print("Retrieved rows:", len(full_coordinate_df))
print("Number of columns:", len(full_coordinate_df.columns))

Expected rows: 434
Retrieved rows: 434
Number of columns: 17


In [26]:
assert len(full_coordinate_df) == total_matching_rows, (
    f"Expected {total_matching_rows} rows, "
    f"but retrieved {len(full_coordinate_df)} rows."
)

print("Complete coordinate result retrieved successfully.")

Complete coordinate result retrieved successfully.


In [27]:
full_identifier_summary = pd.DataFrame(
    {
        "field": [
            "proposal_id",
            "group_ous_uid",
            "member_ous_uid",
            "obs_id",
            "asdm_uid",
            "target_name",
        ],
        "unique_values": [
            full_coordinate_df["proposal_id"].nunique(dropna=True),
            full_coordinate_df["group_ous_uid"].nunique(dropna=True),
            full_coordinate_df["member_ous_uid"].nunique(dropna=True),
            full_coordinate_df["obs_id"].nunique(dropna=True),
            full_coordinate_df["asdm_uid"].nunique(dropna=True),
            full_coordinate_df["target_name"].nunique(dropna=True),
        ],
    }
)

display(full_identifier_summary)

,field,unique_values
0,proposal_id,24
1,group_ous_uid,82
2,member_ous_uid,108
3,obs_id,434
4,asdm_uid,108
5,target_name,8


In [28]:
full_mosaic_summary = (
    full_coordinate_df
    .groupby("is_mosaic", dropna=False)
    .agg(
        archive_row_count=("obs_id", "size"),
        proposal_count=("proposal_id", "nunique"),
        member_ous_count=("member_ous_uid", "nunique"),
        asdm_count=("asdm_uid", "nunique"),
    )
    .reset_index()
)

display(full_mosaic_summary)

,is_mosaic,archive_row_count,proposal_count,member_ous_count,asdm_count
0,F,324,22,79,79
1,T,110,12,29,29


In [29]:
example_frequency_support = (
    full_coordinate_df.loc[
        full_coordinate_df["member_ous_uid"] == example_member_ous,
        "frequency_support",
    ]
    .dropna()
    .unique()
)

print("Number of unique frequency_support values:")
print(len(example_frequency_support))

print("\nComplete frequency_support value:")
print(example_frequency_support[0])

Number of unique frequency_support values:
1

Complete frequency_support value:
[689.28..691.15GHz,976.56kHz,10.8mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [691.10..692.98GHz,976.56kHz,10.8mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [702.98..704.86GHz,976.56kHz,10.5mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [706.65..708.53GHz,976.56kHz,10.5mJy/beam@10km/s,1.2mJy/beam@native, XX YY]


## Validate Archive identifier relationships

The Archive result contains one row per source and spectral window.
Before implementing the reusable Archive client, we verify the relationships
between Member OUS identifiers, ASDM identifiers, mosaic flags, and complete
frequency-support descriptions.

In [30]:
member_ous_summary = (
    full_coordinate_df
    .groupby("member_ous_uid", dropna=False)
    .agg(
        archive_row_count=("obs_id", "size"),
        unique_obs_ids=("obs_id", "nunique"),
        unique_asdm_uids=("asdm_uid", "nunique"),
        unique_group_ous_uids=("group_ous_uid", "nunique"),
        unique_mosaic_values=("is_mosaic", "nunique"),
        unique_frequency_support_values=("frequency_support", "nunique"),
        min_frequency_ghz=("frequency", "min"),
        max_frequency_ghz=("frequency", "max"),
    )
    .reset_index()
)

print("Number of Member OUS groups:", len(member_ous_summary))
display(member_ous_summary.head(20))

Number of Member OUS groups: 108


,member_ous_uid,archive_row_count,unique_obs_ids,unique_asdm_uids,unique_group_ous_uids,unique_mosaic_values,unique_frequency_support_values,min_frequency_ghz,max_frequency_ghz
0,uid://A001/X11a7/X14,4,4,1,1,1,1,213.096375,229.096103
1,uid://A001/X121/X15e,4,4,1,1,1,1,98.812478,112.393430
2,uid://A001/X121/X160,4,4,1,1,1,1,98.786857,112.393428
3,uid://A001/X121/X164,4,4,1,1,1,1,219.168313,234.000020
4,uid://A001/X121/X166,4,4,1,1,1,1,219.168070,234.000047
5,uid://A001/X121/X16c,4,4,1,1,1,1,101.000011,115.064316
6,uid://A001/X121/X16e,4,4,1,1,1,1,101.000006,115.062765
7,uid://A001/X121/X170,4,4,1,1,1,1,101.000017,115.064510
8,uid://A001/X121/X176,4,4,1,1,1,1,230.115098,246.789416
9,uid://A001/X121/X178,4,4,1,1,1,1,230.115100,246.789284


In [31]:
relationship_checks = pd.DataFrame(
    {
        "check": [
            "Member OUS with duplicate obs_id rows",
            "Member OUS linked to multiple ASDM UIDs",
            "Member OUS linked to multiple Group OUS UIDs",
            "Member OUS with multiple mosaic values",
            "Member OUS with multiple frequency_support values",
        ],
        "count": [
            (
                member_ous_summary["archive_row_count"]
                != member_ous_summary["unique_obs_ids"]
            ).sum(),
            (member_ous_summary["unique_asdm_uids"] > 1).sum(),
            (member_ous_summary["unique_group_ous_uids"] > 1).sum(),
            (member_ous_summary["unique_mosaic_values"] > 1).sum(),
            (
                member_ous_summary["unique_frequency_support_values"] > 1
            ).sum(),
        ],
    }
)

display(relationship_checks)

,check,count
0,Member OUS with duplicate obs_id rows,0
1,Member OUS linked to multiple ASDM UIDs,0
2,Member OUS linked to multiple Group OUS UIDs,0
3,Member OUS with multiple mosaic values,0
4,Member OUS with multiple frequency_support values,0


In [32]:
spectral_window_count_distribution = (
    member_ous_summary["archive_row_count"]
    .value_counts()
    .sort_index()
    .rename_axis("archive_rows_per_member_ous")
    .reset_index(name="member_ous_count")
)

display(spectral_window_count_distribution)

,archive_rows_per_member_ous,member_ous_count
0,3,10
1,4,92
2,5,4
3,8,2


In [33]:
member_asdm_pairs = (
    full_coordinate_df[
        ["member_ous_uid", "asdm_uid"]
    ]
    .drop_duplicates()
)

members_per_asdm = (
    member_asdm_pairs
    .groupby("asdm_uid", dropna=False)["member_ous_uid"]
    .nunique()
)

print("Unique Member OUS–ASDM pairs:", len(member_asdm_pairs))
print(
    "ASDM UIDs connected to multiple Member OUS:",
    int((members_per_asdm > 1).sum()),
)

print(
    "Rows where Member OUS UID equals ASDM UID:",
    int(
        (
            full_coordinate_df["member_ous_uid"]
            == full_coordinate_df["asdm_uid"]
        ).sum()
    ),
    "out of",
    len(full_coordinate_df),
)

Unique Member OUS–ASDM pairs: 108
ASDM UIDs connected to multiple Member OUS: 0
Rows where Member OUS UID equals ASDM UID: 0 out of 434
